# Diagnóstico 08 — combinação das otimizações de memória do Dashboard ENEM

## Objetivo

Testar as otimizações identificadas no Diagnóstico 07 em conjunto, mantendo os processos isolados e a equivalência dos dados como critérios obrigatórios.

A sequência é:

1. confirmar o backend real do Memory Pool;
2. testar o arranjo fatorial `use_threads` × `read_dictionary`;
3. comparar Parquets originais e separados por referência usando a melhor combinação encontrada;
4. medir o efeito adicional do downcasting;
5. consolidar a configuração candidata;
6. medir a aplicação real com Gunicorn e um worker.

**O notebook não modifica `app.py`, `dash_utils/features_names.py` nem os Parquets originais.** A integração no loader será uma etapa posterior, após os resultados deste notebook.

### Premissas de comparação

- Cada execução de leitura ocorre em um subprocesso novo.
- As projeções, filtros, shapefile e harmonização municipal seguem o loader atual.
- A métrica principal para memória é o USS após carga/preparação; RSS e tempo são métricas complementares.
- A melhor combinação será escolhida pelo menor pior USS entre ESC e PROVA, para evitar otimizar apenas uma referência.
- A validação por linhas, colunas e hashes usa soma e XOR dos hashes por linha, permitindo comparar conteúdos mesmo quando a ordem física muda.
- Downcasting será avaliado separadamente: os valores inteiros devem permanecer exatos e o erro float32 será explicitado.

> Comece com `REPETICOES = 3`. Se as medianas estiverem próximas ou os intervalos se sobrepuserem muito, aumente para 5 ou mais.


## 0. Configuração do ambiente e localização dos dados

In [1]:
from pathlib import Path
import os
import sys
import json
import time
import gc
import subprocess
import textwrap
import statistics
import itertools
import signal
import urllib.request
import urllib.error
import pandas as pd
import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq
import psutil
from IPython.display import display

# Localiza app_render sem presumir o diretório atual do notebook.
CANDIDATOS_ROOT = [Path.cwd(), *Path.cwd().parents]
CANDIDATOS_ROOT += [
    Path.home() / "PycharmProjects" / "Projeto_ENEM" / "app_render",
]
PROJECT_ROOT = next(
    (p.resolve() for p in CANDIDATOS_ROOT if (p / "dash_utils" / "config.py").exists()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Não encontrei app_render/dash_utils/config.py. "
        "Ajuste PROJECT_ROOT manualmente para a pasta app_render."
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
import dash_utils.config as config

UF = "PA"
REFERENCIAS = ["ESC", "PROVA"]
REPETICOES = 3
TIMEOUT_WORKER = 900

CONSUMO_UF_DIR = Path(config.CONSUMO_DIR) / f"uf={UF}"
DATA_DIR = Path(config.DATA_DIR)
SHAPE_PATH = DATA_DIR / "shapefiles" / f"{UF}_Municipios_2025" / f"{UF}_Municipios_2025.shp"

ARQUIVOS_ORIGINAIS = {
    "painel": CONSUMO_UF_DIR / "painel_municipal.parquet",
    "faixa": CONSUMO_UF_DIR / "faixa_etaria_municipal.parquet",
    "grupo": CONSUMO_UF_DIR / "desempenho_grupos_municipal.parquet",
    "dsp": CONSUMO_UF_DIR / "desempenho_sexo_faixa_etaria_municipal.parquet",
}

# Reutiliza as variantes produzidas pelo Diagnóstico 07, se estiverem disponíveis.
EXPERIMENT_DIR_07 = PROJECT_ROOT / "_diagnosticos_memoria_07"
VARIANT_DIR_07 = EXPERIMENT_DIR_07 / "parquet_variants"
EXPERIMENT_DIR = PROJECT_ROOT / "_diagnosticos_memoria_08"
VARIANT_DIR_08 = EXPERIMENT_DIR / "parquet_variants"
EXPERIMENT_DIR.mkdir(parents=True, exist_ok=True)

COLUNAS = {
    "painel": [
        "ano", "CO_MUNICIPIO_TERRITORIO", "TIPO_REFERENCIA_TERRITORIAL",
        "n_participantes", "n_masculino", "n_feminino",
        "n_faixa_etaria_informada", "n_escola_federal",
        "n_escola_estadual", "n_escola_municipal", "n_escola_privada",
        "n_escola_nao_informada", "media_redacao", "n_validos_redacao",
        "media_objetiva", "n_validos_objetiva", "media_cn", "n_validos_cn",
        "media_ch", "n_validos_ch", "media_lc", "n_validos_lc",
        "media_mt", "n_validos_mt",
    ],
    "faixa": [
        "ano", "CO_MUNICIPIO_TERRITORIO", "TIPO_REFERENCIA_TERRITORIAL",
        "TP_FAIXA_ETARIA", "n_participantes",
    ],
    "grupo": [
        "ano", "CO_MUNICIPIO_TERRITORIO", "TIPO_REFERENCIA_TERRITORIAL",
        "tipo_grupo", "grupo", "disciplina", "media", "n_validos",
        "mediana", "std", "q1", "q3",
    ],
    "dsp": [
        "ano", "CO_MUNICIPIO_TERRITORIO", "TIPO_REFERENCIA_TERRITORIAL",
        "sexo", "faixa_etaria", "disciplina", "n_participantes",
        "n_validos", "media", "std",
    ],
}

faltantes = [str(p) for p in [*ARQUIVOS_ORIGINAIS.values(), SHAPE_PATH] if not p.exists()]
if faltantes:
    raise FileNotFoundError("Arquivos ausentes:\n" + "\n".join(faltantes))

# Detecta as colunas string elegíveis para read_dictionary no schema original.
CANDIDATAS_DICIONARIO = {
    "painel": ["TIPO_REFERENCIA_TERRITORIAL"],
    "faixa": ["TIPO_REFERENCIA_TERRITORIAL"],
    "grupo": ["TIPO_REFERENCIA_TERRITORIAL", "tipo_grupo", "grupo", "disciplina"],
    "dsp": ["TIPO_REFERENCIA_TERRITORIAL", "sexo", "faixa_etaria", "disciplina"],
}
DICTIONARY_COLUMNS = {}
for base, caminho in ARQUIVOS_ORIGINAIS.items():
    schema = pq.read_schema(caminho)
    DICTIONARY_COLUMNS[base] = [
        col for col in CANDIDATAS_DICIONARIO[base]
        if col in schema.names
        and str(schema.field(col).type) in ("string", "large_string", "binary", "large_binary")
    ]

print("PROJECT_ROOT:", PROJECT_ROOT)
print("PyArrow:", pa.__version__)
print("Memory Pool do kernel (não usado para os subprocessos):", pa.default_memory_pool().backend_name)
print("Variantes NB07 disponíveis:", VARIANT_DIR_07.exists())
print("Colunas candidatas a DictionaryArray:", json.dumps(DICTIONARY_COLUMNS, indent=2, ensure_ascii=False))
print("Resultados:", EXPERIMENT_DIR)


PROJECT_ROOT: /home/akel/PycharmProjects/Projeto_ENEM/app_render
PyArrow: 26.0.1
Memory Pool do kernel (não usado para os subprocessos): mimalloc
Variantes NB07 disponíveis: True
Colunas candidatas a DictionaryArray: {
  "painel": [
    "TIPO_REFERENCIA_TERRITORIAL"
  ],
  "faixa": [
    "TIPO_REFERENCIA_TERRITORIAL"
  ],
  "grupo": [
    "TIPO_REFERENCIA_TERRITORIAL",
    "tipo_grupo",
    "grupo",
    "disciplina"
  ],
  "dsp": [
    "TIPO_REFERENCIA_TERRITORIAL",
    "sexo",
    "faixa_etaria",
    "disciplina"
  ]
}
Resultados: /home/akel/PycharmProjects/Projeto_ENEM/app_render/_diagnosticos_memoria_08


## 1. Worker de leitura isolada

O worker é executado em um processo novo. `ARROW_DEFAULT_MEMORY_POOL` é definido no ambiente antes da importação das bibliotecas Arrow. O backend é registrado **dentro do próprio worker**, e uma execução cujo backend real não corresponda ao solicitado será marcada como inválida para a comparação do Memory Pool.

O worker registra:

- USS/RSS após imports, após cada leitura e após preparação territorial;
- USS/RSS antes e depois do downcasting, quando solicitado;
- tamanho profundo de cada DataFrame;
- backend real e métricas do Memory Pool;
- assinaturas dos quatro DataFrames.


In [2]:
WORKER_SOURCE = r'''import os, sys, json, gc, time
from pathlib import Path

payload = json.load(sys.stdin)
project_root = Path(payload["project_root"])
sys.path.insert(0, str(project_root))

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import geopandas as gpd
import psutil
import dash_utils.config as config

UF = payload["uf"]
ref = payload["referencia"]
use_threads = bool(payload.get("use_threads", True))
read_dictionary = bool(payload.get("read_dictionary", False))
layout = payload.get("layout", "original")
apply_downcast = bool(payload.get("apply_downcast", False))
variant_dir = Path(payload["variant_dir"])

cols = {
    "painel": ["ano", "CO_MUNICIPIO_TERRITORIO", "TIPO_REFERENCIA_TERRITORIAL", "n_participantes", "n_masculino", "n_feminino", "n_faixa_etaria_informada", "n_escola_federal", "n_escola_estadual", "n_escola_municipal", "n_escola_privada", "n_escola_nao_informada", "media_redacao", "n_validos_redacao", "media_objetiva", "n_validos_objetiva", "media_cn", "n_validos_cn", "media_ch", "n_validos_ch", "media_lc", "n_validos_lc", "media_mt", "n_validos_mt"],
    "faixa": ["ano", "CO_MUNICIPIO_TERRITORIO", "TIPO_REFERENCIA_TERRITORIAL", "TP_FAIXA_ETARIA", "n_participantes"],
    "grupo": ["ano", "CO_MUNICIPIO_TERRITORIO", "TIPO_REFERENCIA_TERRITORIAL", "tipo_grupo", "grupo", "disciplina", "media", "n_validos", "mediana", "std", "q1", "q3"],
    "dsp": ["ano", "CO_MUNICIPIO_TERRITORIO", "TIPO_REFERENCIA_TERRITORIAL", "sexo", "faixa_etaria", "disciplina", "n_participantes", "n_validos", "media", "std"],
}
originais = {
    "painel": Path(config.CONSUMO_DIR) / f"uf={UF}" / "painel_municipal.parquet",
    "faixa": Path(config.CONSUMO_DIR) / f"uf={UF}" / "faixa_etaria_municipal.parquet",
    "grupo": Path(config.CONSUMO_DIR) / f"uf={UF}" / "desempenho_grupos_municipal.parquet",
    "dsp": Path(config.CONSUMO_DIR) / f"uf={UF}" / "desempenho_sexo_faixa_etaria_municipal.parquet",
}
shape_path = Path(config.DATA_DIR) / "shapefiles" / f"{UF}_Municipios_2025" / f"{UF}_Municipios_2025.shp"
dict_cols = payload.get("dictionary_columns", {})
processo = psutil.Process()

pool = pa.default_memory_pool()
backend_real = pool.backend_name
backend_solicitado = payload.get("memory_pool")
backend_confirmado = (backend_solicitado is None) or (backend_real == backend_solicitado)

def memoria():
    m = processo.memory_full_info()
    return {"rss_mb": m.rss / 1024**2, "uss_mb": getattr(m, "uss", float("nan")) / 1024**2}

def snapshot(etapa):
    return {"etapa": etapa, **memoria(), "arrow_pool_mb": pool.bytes_allocated() / 1024**2}

def normalizar_nome(nome):
    import unicodedata
    if not isinstance(nome, str):
        return nome
    return unicodedata.normalize("NFKD", nome).encode("ascii", "ignore").decode("ascii").upper().strip()

def assinatura(df):
    h = pd.util.hash_pandas_object(df, index=False).to_numpy(dtype=np.uint64)
    return {
        "linhas": int(len(df)),
        "colunas": list(df.columns),
        "dtypes": {col: str(dtype) for col, dtype in df.dtypes.items()},
        "hash_soma_uint64": int(h.sum(dtype=np.uint64)),
        "hash_xor_uint64": int(np.bitwise_xor.reduce(h, initial=np.uint64(0))),
    }

def caminho_leitura(base):
    if layout == "original":
        return originais[base], True
    if layout == "separado_rg_100000":
        return variant_dir / base / "separado_rg_100000" / f"{ref}.parquet", False
    raise ValueError(f"Layout desconhecido: {layout}")

inicio_total = time.perf_counter()
medidas = [snapshot("baseline_apos_imports")]
frames = {}

for base in ["painel", "faixa", "grupo", "dsp"]:
    caminho, aplicar_filtro = caminho_leitura(base)
    if not caminho.exists():
        raise FileNotFoundError(f"Parquet ausente para {base}/{ref}: {caminho}")
    filtros = None
    if aplicar_filtro:
        valores = ["ESC", "RESIDENCIA"] if ref == "ESC" else ["PROVA"]
        filtros = [("TIPO_REFERENCIA_TERRITORIAL", "in", valores)]
    kwargs = {"columns": cols[base], "use_threads": use_threads, "filters": filtros}
    if read_dictionary:
        # Só solicita DictionaryArray para colunas presentes no arquivo lido.
        schema_names = pq.read_schema(caminho).names
        selecionadas = [c for c in dict_cols.get(base, []) if c in schema_names]
        if selecionadas:
            kwargs["read_dictionary"] = selecionadas
    frames[base] = pd.read_parquet(caminho, engine="pyarrow", **kwargs)
    medidas.append(snapshot(f"apos_leitura_{base}"))

# Repete a preparação territorial utilizada pelo loader atual.
shape = gpd.read_file(shape_path)
mapa = {
    str(cd): normalizar_nome(nm)
    for cd, nm in zip(shape["CD_MUN"], shape["NM_MUN"])
}
for base, df in list(frames.items()):
    df["TIPO_REFERENCIA_TERRITORIAL"] = df["TIPO_REFERENCIA_TERRITORIAL"].replace("RESIDENCIA", "ESC")
    df = df.loc[df["TIPO_REFERENCIA_TERRITORIAL"].eq(ref)].copy()
    df["NO_MUNICIPIO_TERRITORIO"] = df["CO_MUNICIPIO_TERRITORIO"].astype(str).map(mapa)
    df = df.dropna(subset=["NO_MUNICIPIO_TERRITORIO"]).reset_index(drop=True)
    frames[base] = df
medidas.append(snapshot("apos_shapefile_e_preparacao"))

uss_pre_downcast = memoria()["uss_mb"]
deep_antes = {nome: float(df.memory_usage(deep=True).sum() / 1024**2) for nome, df in frames.items()}
downcast_info = {}

if apply_downcast:
    for nome, df in frames.items():
        alteracoes = []
        erros_float = []
        for coluna in df.columns:
            serie = df[coluna]
            dtype_original = str(serie.dtype)
            if pd.api.types.is_integer_dtype(serie.dtype) and dtype_original in ("int64", "uint64"):
                nova = pd.to_numeric(serie, downcast="integer")
                if str(nova.dtype) != dtype_original:
                    df[coluna] = nova
                    alteracoes.append({"coluna": coluna, "de": dtype_original, "para": str(nova.dtype), "erro_abs_max": 0.0})
            elif pd.api.types.is_float_dtype(serie.dtype) and dtype_original == "float64":
                original = serie.to_numpy(dtype="float64", copy=True)
                nova = serie.astype("float32")
                if len(original):
                    diferencas = np.abs(original - nova.to_numpy(dtype="float64"))
                    erro = float(np.nanmax(diferencas)) if np.isfinite(diferencas).any() else 0.0
                else:
                    erro = 0.0
                df[coluna] = nova
                alteracoes.append({"coluna": coluna, "de": dtype_original, "para": "float32", "erro_abs_max": erro})
                erros_float.append({"coluna": coluna, "erro_abs_max": erro})
                del original, nova
        downcast_info[nome] = {"alteracoes": alteracoes, "erros_float": erros_float}
    medidas.append(snapshot("apos_downcast"))

metricas_apos_carga = snapshot("apos_carga_pre_validacao")
deep_depois = {nome: float(df.memory_usage(deep=True).sum() / 1024**2) for nome, df in frames.items()}
assinaturas = {nome: assinatura(df) for nome, df in frames.items()}

for nome in list(frames):
    del frames[nome]
del frames, shape, mapa
gc.collect()
medidas.append(snapshot("apos_del_gc"))

pool_info = {
    "backend_real": backend_real,
    "backend_solicitado": backend_solicitado,
    "backend_confirmado": backend_confirmado,
    "bytes_allocated_mb": pool.bytes_allocated() / 1024**2,
    "max_memory_mb": pool.max_memory() / 1024**2 if hasattr(pool, "max_memory") else None,
    "environment": os.environ.get("ARROW_DEFAULT_MEMORY_POOL"),
}
resultado = {
    "status": "ok",
    "uf": UF,
    "referencia": ref,
    "layout": layout,
    "use_threads": use_threads,
    "read_dictionary": read_dictionary,
    "apply_downcast": apply_downcast,
    "pool": pool_info,
    "tempo_total_s": time.perf_counter() - inicio_total,
    "medidas": medidas,
    "apos_carga": metricas_apos_carga,
    "uss_pre_downcast_mb": uss_pre_downcast,
    "deep_antes_mb": deep_antes,
    "deep_depois_mb": deep_depois,
    "downcast": downcast_info,
    "assinaturas": assinaturas,
}
print(json.dumps(resultado, ensure_ascii=False))'''
WORKER_SCRIPT = textwrap.dedent(WORKER_SOURCE)

def executar_worker(
    referencia,
    use_threads=True,
    read_dictionary=False,
    memory_pool=None,
    layout="original",
    apply_downcast=False,
    timeout=TIMEOUT_WORKER,
):
    payload = {
        "project_root": str(PROJECT_ROOT),
        "variant_dir": str(VARIANT_DIR),
        "uf": UF,
        "referencia": referencia,
        "use_threads": use_threads,
        "read_dictionary": read_dictionary,
        "dictionary_columns": DICTIONARY_COLUMNS,
        "memory_pool": memory_pool,
        "layout": layout,
        "apply_downcast": apply_downcast,
    }
    env = os.environ.copy()
    env["PYTHONUNBUFFERED"] = "1"
    if memory_pool:
        env["ARROW_DEFAULT_MEMORY_POOL"] = memory_pool
    else:
        env.pop("ARROW_DEFAULT_MEMORY_POOL", None)

    proc = subprocess.run(
        [sys.executable, "-c", WORKER_SCRIPT],
        input=json.dumps(payload), text=True, capture_output=True,
        cwd=PROJECT_ROOT, env=env, timeout=timeout,
    )
    if proc.returncode != 0:
        raise RuntimeError(
            f"Worker falhou ({proc.returncode}) em {referencia}/{layout}.\n"
            f"STDOUT:\n{proc.stdout[-4000:]}\nSTDERR:\n{proc.stderr[-6000:]}"
        )
    linhas_json = [linha for linha in proc.stdout.splitlines() if linha.strip().startswith("{")]
    if not linhas_json:
        raise RuntimeError(f"Worker não retornou JSON.\n{proc.stdout}\n{proc.stderr}")
    return json.loads(linhas_json[-1])

def resumir_execucoes(resultados):
    linhas = []
    for r in resultados:
        linhas.append({
            "cenario": r.get("cenario", "sem_nome"),
            "referencia": r["referencia"],
            "repeticao": r.get("repeticao"),
            "USS_apos_carga_MB": r["apos_carga"]["uss_mb"],
            "RSS_apos_carga_MB": r["apos_carga"]["rss_mb"],
            "USS_apos_gc_MB": r["medidas"][-1]["uss_mb"],
            "Arrow_pool_MB": r["pool"]["bytes_allocated_mb"],
            "tempo_total_s": r["tempo_total_s"],
            "deep_total_MB": sum(r["deep_depois_mb"].values()),
            "pool_real": r["pool"]["backend_real"],
            "pool_confirmado": r["pool"]["backend_confirmado"],
        })
    detalhe = pd.DataFrame(linhas)
    if detalhe.empty:
        return detalhe, pd.DataFrame()
    resumo = (
        detalhe.groupby(["cenario", "referencia"], as_index=False)
        .agg(
            USS_mediana_MB=("USS_apos_carga_MB", "median"),
            USS_min_MB=("USS_apos_carga_MB", "min"),
            USS_max_MB=("USS_apos_carga_MB", "max"),
            RSS_mediana_MB=("RSS_apos_carga_MB", "median"),
            USS_apos_gc_mediana_MB=("USS_apos_gc_MB", "median"),
            Arrow_pool_mediana_MB=("Arrow_pool_MB", "median"),
            tempo_mediano_s=("tempo_total_s", "median"),
            deep_total_mediana_MB=("deep_total_MB", "median"),
            n=("repeticao", "count"),
        )
    )
    return detalhe, resumo

def validar_assinaturas(resultado_base, resultado_teste):
    saida = []
    for base in ["painel", "faixa", "grupo", "dsp"]:
        a = resultado_base["assinaturas"][base]
        b = resultado_teste["assinaturas"][base]
        iguais = (
            a["linhas"] == b["linhas"]
            and a["colunas"] == b["colunas"]
            and a["hash_soma_uint64"] == b["hash_soma_uint64"]
            and a["hash_xor_uint64"] == b["hash_xor_uint64"]
        )
        saida.append({
            "base": base,
            "equivalente": iguais,
            "equivalente_valores_e_estrutura": iguais,
            "dtype_alterou": a.get("dtypes") != b.get("dtypes"),
            "dtypes_base": json.dumps(a.get("dtypes", {}), ensure_ascii=False),
            "dtypes_teste": json.dumps(b.get("dtypes", {}), ensure_ascii=False),
            "linhas_base": a["linhas"],
            "linhas_teste": b["linhas"],
        })
    return saida


## 2. Experimento 1 — confirmar o Memory Pool

Este teste mantém leitura e dados constantes (`use_threads=False`, sem `read_dictionary`, Parquets originais) e altera apenas o backend solicitado. O campo `backend_real` vem do próprio processo que fez a leitura.

Um resultado só será considerado válido quando `backend_real == backend_solicitado`. Se `jemalloc` não estiver disponível na compilação do PyArrow, ele será marcado como não confirmado e não entrará na seleção.


In [3]:
POOL_CANDIDATOS = ["mimalloc", "system", "jemalloc"]
POOL_TEST_THREADS = False
resultados_pool = []

for backend_solicitado in POOL_CANDIDATOS:
    for ref in REFERENCIAS:
        for repeticao in range(1, REPETICOES + 1):
            print(f"Pool={backend_solicitado} | {ref} | {repeticao}/{REPETICOES}")
            r = executar_worker(
                referencia=ref,
                use_threads=POOL_TEST_THREADS,
                read_dictionary=False,
                memory_pool=backend_solicitado,
                layout="original",
            )
            r["cenario"] = f"pool_{backend_solicitado}"
            r["pool_solicitado"] = backend_solicitado
            r["repeticao"] = repeticao
            resultados_pool.append(r)

pool_linhas = []
for r in resultados_pool:
    pool_linhas.append({
        "pool_solicitado": r["pool_solicitado"],
        "pool_real": r["pool"]["backend_real"],
        "backend_confirmado": r["pool"]["backend_confirmado"],
        "referencia": r["referencia"],
        "repeticao": r["repeticao"],
        "USS_apos_carga_MB": r["apos_carga"]["uss_mb"],
        "RSS_apos_carga_MB": r["apos_carga"]["rss_mb"],
        "USS_apos_gc_MB": r["medidas"][-1]["uss_mb"],
        "Arrow_pool_apos_gc_MB": r["pool"]["bytes_allocated_mb"],
        "Arrow_pool_pico_MB": r["pool"]["max_memory_mb"],
        "tempo_total_s": r["tempo_total_s"],
    })
df_pool_detalhe = pd.DataFrame(pool_linhas)
display(df_pool_detalhe)

pool_validos = df_pool_detalhe.loc[df_pool_detalhe["backend_confirmado"]].copy()
if pool_validos.empty:
    raise RuntimeError("Nenhum Memory Pool foi confirmado. Verifique a instalação do PyArrow.")
pool_resumo = (pool_validos.groupby(["pool_real", "referencia"], as_index=False)
    .agg(USS_mediana_MB=("USS_apos_carga_MB", "median"),
         RSS_mediana_MB=("RSS_apos_carga_MB", "median"),
         tempo_mediano_s=("tempo_total_s", "median"),
         n=("repeticao", "count")))
display(pool_resumo)

# Seleciona o backend com menor pior USS entre as duas referências.
pool_selecao = (pool_resumo.groupby("pool_real", as_index=False)
    .agg(USS_pior_referencia_MB=("USS_mediana_MB", "max"),
         USS_medio_referencias_MB=("USS_mediana_MB", "mean"),
         tempo_medio_s=("tempo_mediano_s", "mean"))
    .sort_values(["USS_pior_referencia_MB", "tempo_medio_s"]))
display(pool_selecao)
POOL_ESCOLHIDO = str(pool_selecao.iloc[0]["pool_real"])
print("Pool selecionado automaticamente:", POOL_ESCOLHIDO)


Pool=mimalloc | ESC | 1/3


NameError: name 'VARIANT_DIR' is not defined

## 3. Experimento 2 — arranjo fatorial `use_threads` × `read_dictionary`

Serão testadas as quatro combinações, usando o mesmo Memory Pool confirmado na etapa anterior. Isso permite observar se os ganhos de threads e DictionaryArray são complementares ou se uma otimização altera o efeito da outra.

A seleção automática considera o menor **pior USS mediano** entre ESC e PROVA. O tempo médio funciona como critério secundário. Se preferir outra combinação, altere `COMBINACAO_ESCOLHIDA` depois de examinar os resultados.


In [ ]:
COMBINACOES = [
    {"cenario": "threads_true_dictionary_false", "use_threads": True,  "read_dictionary": False},
    {"cenario": "threads_false_dictionary_false", "use_threads": False, "read_dictionary": False},
    {"cenario": "threads_true_dictionary_true",  "use_threads": True,  "read_dictionary": True},
    {"cenario": "threads_false_dictionary_true", "use_threads": False, "read_dictionary": True},
]

resultados_fatorial = []
for combinacao in COMBINACOES:
    for ref in REFERENCIAS:
        for repeticao in range(1, REPETICOES + 1):
            print(f"{combinacao['cenario']} | {ref} | {repeticao}/{REPETICOES}")
            r = executar_worker(
                referencia=ref,
                use_threads=combinacao["use_threads"],
                read_dictionary=combinacao["read_dictionary"],
                memory_pool=POOL_ESCOLHIDO,
                layout="original",
            )
            r["cenario"] = combinacao["cenario"]
            r["repeticao"] = repeticao
            resultados_fatorial.append(r)

fatorial_detalhe, fatorial_resumo = resumir_execucoes(resultados_fatorial)
display(fatorial_detalhe)
display(fatorial_resumo)

fatorial_selecao = (fatorial_resumo.groupby("cenario", as_index=False)
    .agg(USS_pior_referencia_MB=("USS_mediana_MB", "max"),
         USS_medio_referencias_MB=("USS_mediana_MB", "mean"),
         RSS_pior_referencia_MB=("RSS_mediana_MB", "max"),
         tempo_medio_s=("tempo_mediano_s", "mean"))
    .sort_values(["USS_pior_referencia_MB", "tempo_medio_s"]))
display(fatorial_selecao)

COMBINACAO_ESCOLHIDA = str(fatorial_selecao.iloc[0]["cenario"])
PARAMETROS_COMBINACAO = next(x for x in COMBINACOES if x["cenario"] == COMBINACAO_ESCOLHIDA)
print("Combinação selecionada:", PARAMETROS_COMBINACAO)

# Equivalência de cada configuração com a leitura original padrão por referência.
validacoes_fatorial = []
for ref in REFERENCIAS:
    baseline = next(r for r in resultados_fatorial
                    if r["referencia"] == ref
                    and r["cenario"] == "threads_true_dictionary_false"
                    and r["repeticao"] == 1)
    for cenario in [x["cenario"] for x in COMBINACOES]:
        teste = next(r for r in resultados_fatorial
                     if r["referencia"] == ref and r["cenario"] == cenario
                     and r["repeticao"] == 1)
        for item in validar_assinaturas(baseline, teste):
            validacoes_fatorial.append({"referencia": ref, "cenario": cenario, **item})
df_validacao_fatorial = pd.DataFrame(validacoes_fatorial)
display(df_validacao_fatorial)
if not df_validacao_fatorial["equivalente"].all():
    print("ATENÇÃO: pelo menos uma combinação não passou na validação de equivalência.")
else:
    print("Todas as combinações mantiveram linhas, colunas e assinaturas equivalentes.")


## 4. Experimento 3 — configuração escolhida × Parquets separados

Com a melhor combinação da etapa 3, compara-se:

- `original`: arquivo atual com filtro ESC/PROVA durante a leitura;
- `separado_rg_100000`: arquivo específico para cada referência, sem filtro territorial de leitura.

O notebook reutiliza as variantes do Diagnóstico 07 quando elas existem. Se não existirem, a célula seguinte as gera em uma pasta própria do Diagnóstico 08. Os arquivos originais nunca são sobrescritos.


In [ ]:
# Reutiliza NB07 quando disponível; caso contrário, usa a pasta de variantes do NB08.
def variantes_separadas_completas(pasta):
    return all(
        (pasta / base / "separado_rg_100000" / f"{ref}.parquet").exists()
        for base in ARQUIVOS_ORIGINAIS for ref in REFERENCIAS
    )

if variantes_separadas_completas(VARIANT_DIR_07):
    VARIANT_DIR = VARIANT_DIR_07
    print("Reutilizando variantes do Diagnóstico 07:", VARIANT_DIR)
else:
    VARIANT_DIR = VARIANT_DIR_08
    print("Gerando variantes separadas no Diagnóstico 08:", VARIANT_DIR)
    for base, origem in ARQUIVOS_ORIGINAIS.items():
        destino_dir = VARIANT_DIR / base / "separado_rg_100000"
        destino_dir.mkdir(parents=True, exist_ok=True)
        df = pd.read_parquet(origem, engine="pyarrow", columns=COLUNAS[base], use_threads=False)
        df["TIPO_REFERENCIA_TERRITORIAL"] = df["TIPO_REFERENCIA_TERRITORIAL"].replace("RESIDENCIA", "ESC")
        df = df.loc[df["TIPO_REFERENCIA_TERRITORIAL"].isin(REFERENCIAS)].copy()
        for ref in REFERENCIAS:
            parte = df.loc[df["TIPO_REFERENCIA_TERRITORIAL"].eq(ref)]
            destino = destino_dir / f"{ref}.parquet"
            parte.to_parquet(
                destino, engine="pyarrow", index=False,
                compression="snappy", use_dictionary=True,
                row_group_size=100000,
            )
            print(base, ref, "linhas:", len(parte), "arquivo:", destino)
        del parte, df
        gc.collect()

if not variantes_separadas_completas(VARIANT_DIR):
    raise FileNotFoundError("Não foi possível localizar/gerar todos os Parquets separados.")

resultados_layout_combinado = []
for layout in ["original", "separado_rg_100000"]:
    for ref in REFERENCIAS:
        for repeticao in range(1, REPETICOES + 1):
            print(f"{layout} | {ref} | {repeticao}/{REPETICOES}")
            r = executar_worker(
                referencia=ref,
                use_threads=PARAMETROS_COMBINACAO["use_threads"],
                read_dictionary=PARAMETROS_COMBINACAO["read_dictionary"],
                memory_pool=POOL_ESCOLHIDO,
                layout=layout,
            )
            r["cenario"] = layout
            r["repeticao"] = repeticao
            resultados_layout_combinado.append(r)

layout_detalhe, layout_resumo = resumir_execucoes(resultados_layout_combinado)
display(layout_detalhe)
display(layout_resumo)

layout_selecao = (layout_resumo.groupby("cenario", as_index=False)
    .agg(USS_pior_referencia_MB=("USS_mediana_MB", "max"),
         USS_medio_referencias_MB=("USS_mediana_MB", "mean"),
         RSS_pior_referencia_MB=("RSS_mediana_MB", "max"),
         tempo_medio_s=("tempo_mediano_s", "mean"))
    .sort_values(["USS_pior_referencia_MB", "tempo_medio_s"]))
display(layout_selecao)
LAYOUT_ESCOLHIDO = str(layout_selecao.iloc[0]["cenario"])
print("Layout selecionado:", LAYOUT_ESCOLHIDO)

# Compara assinatura do layout separado com a leitura original, mesma referência.
validacoes_layout = []
for ref in REFERENCIAS:
    base = next(r for r in resultados_layout_combinado
                if r["referencia"] == ref and r["layout"] == "original" and r["repeticao"] == 1)
    teste = next(r for r in resultados_layout_combinado
                 if r["referencia"] == ref and r["layout"] == "separado_rg_100000" and r["repeticao"] == 1)
    for item in validar_assinaturas(base, teste):
        validacoes_layout.append({"referencia": ref, **item})
df_validacao_layout = pd.DataFrame(validacoes_layout)
display(df_validacao_layout)
if not df_validacao_layout["equivalente"].all():
    raise AssertionError("A variante separada não passou na validação de equivalência.")


## 5. Experimento 4 — efeito adicional do downcasting

Aqui o melhor pool, a melhor combinação de threads/dicionário e o melhor layout já selecionados são mantidos constantes. Comparam-se DataFrames sem e com downcasting.

O worker mede USS e tamanho profundo antes e depois da conversão. Também registra as alterações de dtype e o erro absoluto máximo dos floats convertidos para `float32`.

O downcasting não será considerado automaticamente aprovado: confira o erro em relação à precisão exigida pelos indicadores e gráficos.


In [ ]:
resultados_downcast_combinado = []
for aplicar in [False, True]:
    cenario = "com_downcast" if aplicar else "sem_downcast"
    for ref in REFERENCIAS:
        for repeticao in range(1, REPETICOES + 1):
            print(f"{cenario} | {ref} | {repeticao}/{REPETICOES}")
            r = executar_worker(
                referencia=ref,
                use_threads=PARAMETROS_COMBINACAO["use_threads"],
                read_dictionary=PARAMETROS_COMBINACAO["read_dictionary"],
                memory_pool=POOL_ESCOLHIDO,
                layout=LAYOUT_ESCOLHIDO,
                apply_downcast=aplicar,
            )
            r["cenario"] = cenario
            r["repeticao"] = repeticao
            resultados_downcast_combinado.append(r)

# Resumo focado em memória antes/depois do downcasting.
linhas_downcast_combinado = []
for r in resultados_downcast_combinado:
    linhas_downcast_combinado.append({
        "cenario": r["cenario"],
        "referencia": r["referencia"],
        "repeticao": r["repeticao"],
        "USS_antes_downcast_MB": r["uss_pre_downcast_mb"],
        "USS_apos_carga_MB": r["apos_carga"]["uss_mb"],
        "deep_antes_MB": sum(r["deep_antes_mb"].values()),
        "deep_depois_MB": sum(r["deep_depois_mb"].values()),
        "pool_real": r["pool"]["backend_real"],
    })
df_downcast_combinado_detalhe = pd.DataFrame(linhas_downcast_combinado)
display(df_downcast_combinado_detalhe)

resumo_downcast_combinado = (df_downcast_combinado_detalhe
    .groupby(["cenario", "referencia"], as_index=False)
    .agg(USS_mediana_MB=("USS_apos_carga_MB", "median"),
         USS_antes_downcast_mediana_MB=("USS_antes_downcast_MB", "median"),
         deep_antes_mediana_MB=("deep_antes_MB", "median"),
         deep_depois_mediana_MB=("deep_depois_MB", "median")))
resumo_downcast_combinado["reducao_deep_pct"] = (
    1 - resumo_downcast_combinado["deep_depois_mediana_MB"] /
    resumo_downcast_combinado["deep_antes_mediana_MB"]
) * 100
display(resumo_downcast_combinado)

TOLERANCIA_FLOAT32 = 1e-4
linhas_erros = []
for r in resultados_downcast_combinado:
    if not r["apply_downcast"]:
        continue
    for base, info in r["downcast"].items():
        for item in info["erros_float"]:
            linhas_erros.append({
                "referencia": r["referencia"],
                "repeticao": r["repeticao"],
                "base": base,
                "coluna": item["coluna"],
                "erro_abs_max": item["erro_abs_max"],
                "dentro_tolerancia": item["erro_abs_max"] <= TOLERANCIA_FLOAT32,
            })
df_erros_downcast = pd.DataFrame(linhas_erros)
display(df_erros_downcast)
if not df_erros_downcast.empty:
    print("Todos os erros dentro da tolerância?", bool(df_erros_downcast["dentro_tolerancia"].all()))


## 6. Consolidação da configuração candidata

A configuração candidata é formada pelo backend confirmado, combinação fatorial escolhida e layout escolhido. O downcasting fica como uma decisão separada, pois reduz principalmente o tamanho profundo dos DataFrames e pode afetar a precisão numérica.

A tabela final compara a configuração selecionada com a leitura original padrão (`threads=True`, sem dictionary, Memory Pool padrão, layout original), usando as medições deste notebook. A comparação é válida somente quando as assinaturas forem equivalentes.


In [ ]:
# Baseline do fatorial: threads=True, sem dictionary, usando o pool selecionado.
# A baseline de deploy anterior ao notebook pode ter outro Memory Pool; por isso,
# esta comparação isola threads/dictionary/layout sob o mesmo backend confirmado.
baseline_fatorial = fatorial_resumo.loc[
    fatorial_resumo["cenario"].eq("threads_true_dictionary_false")
].copy()
candidato_final = layout_resumo.loc[
    layout_resumo["cenario"].eq(LAYOUT_ESCOLHIDO)
].copy()
comparacao_final = baseline_fatorial.merge(
    candidato_final, on="referencia", suffixes=("_baseline", "_candidato")
)
comparacao_final["reducao_USS_pct"] = (
    1 - comparacao_final["USS_mediana_MB_candidato"] /
    comparacao_final["USS_mediana_MB_baseline"]
) * 100
comparacao_final["reducao_RSS_pct"] = (
    1 - comparacao_final["RSS_mediana_MB_candidato"] /
    comparacao_final["RSS_mediana_MB_baseline"]
) * 100
comparacao_final["reducao_tempo_pct"] = (
    1 - comparacao_final["tempo_mediano_s_candidato"] /
    comparacao_final["tempo_mediano_s_baseline"]
) * 100
display(comparacao_final)

downcast_aprovado = (
    df_erros_downcast.empty
    or bool(df_erros_downcast["dentro_tolerancia"].all())
)
CONFIGURACAO_CANDIDATA = {
    "uf": UF,
    "memory_pool": POOL_ESCOLHIDO,
    "use_threads": PARAMETROS_COMBINACAO["use_threads"],
    "read_dictionary": PARAMETROS_COMBINACAO["read_dictionary"],
    "layout": LAYOUT_ESCOLHIDO,
    "downcast_aprovado_na_tolerancia_configurada": downcast_aprovado,
}
print(json.dumps(CONFIGURACAO_CANDIDATA, indent=2, ensure_ascii=False))

if not df_validacao_fatorial["equivalente"].all() or not df_validacao_layout["equivalente"].all():
    print("CONFIGURAÇÃO NÃO APROVADA: existe divergência na validação dos dados.")
else:
    print("Validação de conteúdo aprovada para as combinações e layouts testados.")


## 7. Experimento 5 — medição da aplicação real com Gunicorn e Dash

Esta etapa inicia a aplicação atual sem alterar o código de produção. O Memory Pool selecionado é passado ao ambiente do processo antes da importação do aplicativo.

**Limitação importante:** `use_threads` e `read_dictionary` são opções de leitura do loader. Como esta etapa executa o `app.py` atual, elas só terão efeito na aplicação se já estiverem implementadas em `dash_utils/features_names.py`. Este teste mede o impacto do Memory Pool e a arquitetura real de processos; não simula automaticamente alterações ainda não integradas ao loader.

O servidor permanece desabilitado por padrão. Antes de habilitar, confirme que a porta 8050 está livre e que o comando `gunicorn app:server` funciona no ambiente `app_render`.


In [ ]:
GUNICORN_HOST = "127.0.0.1"
GUNICORN_PORT = 8050
GUNICORN_WORKERS = 1
GUNICORN_ENABLE = False
GUNICORN_POOL = POOL_ESCOLHIDO
GUNICORN_LOG = EXPERIMENT_DIR / "gunicorn_diagnostico08.log"

print("Gunicorn habilitado:", GUNICORN_ENABLE)
print("Pool do processo:", GUNICORN_POOL)
print("Comando:", f"{sys.executable} -m gunicorn app:server --bind {GUNICORN_HOST}:{GUNICORN_PORT} --workers {GUNICORN_WORKERS}")

def memoria_arvore_processo(pid):
    try:
        raiz = psutil.Process(pid)
        processos = [raiz] + raiz.children(recursive=True)
    except psutil.Error:
        return []
    saida = []
    for proc in processos:
        try:
            m = proc.memory_full_info()
            saida.append({
                "pid": proc.pid,
                "nome": proc.name(),
                "rss_mb": m.rss / 1024**2,
                "uss_mb": getattr(m, "uss", float("nan")) / 1024**2,
                "pss_mb": getattr(m, "pss", float("nan")) / 1024**2,
                "status": proc.status(),
            })
        except (psutil.NoSuchProcess, psutil.AccessDenied):
            pass
    return saida

def totais_arvore(medidas):
    return {
        "n_processos": len(medidas),
        "RSS_soma_MB": sum(x["rss_mb"] for x in medidas),
        "USS_soma_MB": sum(x["uss_mb"] for x in medidas),
        "PSS_soma_MB": sum(x["pss_mb"] for x in medidas if np.isfinite(x["pss_mb"])),
    }

def iniciar_gunicorn(tempo_espera=60):
    if not GUNICORN_ENABLE:
        print("Execução desabilitada. Altere GUNICORN_ENABLE = True para iniciar.")
        return None
    comando = [
        sys.executable, "-m", "gunicorn", "app:server",
        "--bind", f"{GUNICORN_HOST}:{GUNICORN_PORT}",
        "--workers", str(GUNICORN_WORKERS),
        "--timeout", "120",
        "--access-logfile", "-", "--error-logfile", "-",
    ]
    env = os.environ.copy()
    env["PYTHONUNBUFFERED"] = "1"
    env["ARROW_DEFAULT_MEMORY_POOL"] = GUNICORN_POOL
    with open(GUNICORN_LOG, "w", encoding="utf-8") as log:
        proc = subprocess.Popen(comando, cwd=PROJECT_ROOT, stdout=log,
                                stderr=subprocess.STDOUT, env=env)
    inicio = time.time()
    pronto = False
    while time.time() - inicio < tempo_espera:
        if proc.poll() is not None:
            raise RuntimeError(f"Gunicorn encerrou ({proc.returncode}). Consulte {GUNICORN_LOG}")
        try:
            with urllib.request.urlopen(f"http://{GUNICORN_HOST}:{GUNICORN_PORT}/", timeout=2) as resp:
                if 200 <= resp.status < 500:
                    pronto = True
                    break
        except Exception:
            time.sleep(1)
    if not pronto:
        proc.terminate()
        raise TimeoutError(f"Gunicorn não respondeu em {tempo_espera}s. Consulte {GUNICORN_LOG}")
    time.sleep(3)
    medidas = memoria_arvore_processo(proc.pid)
    return {"process": proc, "medidas": medidas, "totais": totais_arvore(medidas), "log": str(GUNICORN_LOG)}

teste_gunicorn = iniciar_gunicorn()
if teste_gunicorn:
    display(pd.DataFrame(teste_gunicorn["medidas"]))
    display(pd.DataFrame([teste_gunicorn["totais"]]))
    print("Log:", teste_gunicorn["log"])


### Medição durante navegação manual

Com o servidor ativo, abra `http://127.0.0.1:8050`, navegue pelas páginas e acione os gráficos/callbacks que carregam `painel`, `faixa`, `grupo` e `dsp`. Execute a célula abaixo enquanto navega para registrar a série temporal.


In [ ]:
def capturar_memoria_gunicorn(duracao_s=60, intervalo_s=1):
    if "teste_gunicorn" not in globals() or not teste_gunicorn:
        print("Nenhum Gunicorn iniciado por este notebook.")
        return pd.DataFrame()
    proc = teste_gunicorn["process"]
    registros = []
    inicio = time.time()
    while time.time() - inicio < duracao_s and proc.poll() is None:
        medidas = memoria_arvore_processo(proc.pid)
        total = totais_arvore(medidas)
        registros.append({"tempo_decorrido_s": time.time() - inicio, **total})
        time.sleep(intervalo_s)
    df = pd.DataFrame(registros)
    if not df.empty:
        display(df.describe().T)
        print("Pico RSS (soma processos):", round(df["RSS_soma_MB"].max(), 2), "MB")
        print("Pico USS (soma processos):", round(df["USS_soma_MB"].max(), 2), "MB")
    return df

# Execute enquanto navega no dashboard:
# memoria_gunicorn_navegacao = capturar_memoria_gunicorn(duracao_s=90, intervalo_s=1)


### Encerramento do servidor

In [ ]:
if "teste_gunicorn" in globals() and teste_gunicorn:
    proc_gunicorn = teste_gunicorn["process"]
    if proc_gunicorn.poll() is None:
        proc_gunicorn.terminate()
        try:
            proc_gunicorn.wait(timeout=10)
        except subprocess.TimeoutExpired:
            proc_gunicorn.kill()
            proc_gunicorn.wait(timeout=5)
        print("Gunicorn encerrado.")
    else:
        print("Gunicorn já estava encerrado.")
else:
    print("Nenhum processo Gunicorn ativo neste notebook.")


## 8. Exportação dos resultados

A célula exporta apenas os conjuntos que foram efetivamente executados. Os arquivos são gravados em `_diagnosticos_memoria_08`.


In [ ]:
ARQUIVOS_EXPORTADOS = []

def exportar_df(nome, df):
    if df is None or len(df) == 0:
        print(f"Ignorado (sem dados): {nome}")
        return
    caminho = EXPERIMENT_DIR / f"{nome}.csv"
    df.to_csv(caminho, index=False)
    ARQUIVOS_EXPORTADOS.append(str(caminho))
    print(caminho)

for nome, variavel in [
    ("memory_pool_detalhe", "df_pool_detalhe"),
    ("memory_pool_resumo", "pool_resumo"),
    ("fatorial_detalhe", "fatorial_detalhe"),
    ("fatorial_resumo", "fatorial_resumo"),
    ("fatorial_selecao", "fatorial_selecao"),
    ("validacao_fatorial", "df_validacao_fatorial"),
    ("layout_combinado_detalhe", "layout_detalhe"),
    ("layout_combinado_resumo", "layout_resumo"),
    ("layout_selecao", "layout_selecao"),
    ("validacao_layout_combinado", "df_validacao_layout"),
    ("downcast_combinado_detalhe", "df_downcast_combinado_detalhe"),
    ("downcast_combinado_resumo", "resumo_downcast_combinado"),
    ("downcast_erros", "df_erros_downcast"),
    ("comparacao_configuracao_final", "comparacao_final"),
]:
    if variavel in globals():
        exportar_df(nome, globals()[variavel])

if "memoria_gunicorn_navegacao" in globals() and len(memoria_gunicorn_navegacao):
    exportar_df("gunicorn_memoria_navegacao", memoria_gunicorn_navegacao)

if "CONFIGURACAO_CANDIDATA" in globals():
    caminho_json = EXPERIMENT_DIR / "configuracao_candidata.json"
    caminho_json.write_text(json.dumps(CONFIGURACAO_CANDIDATA, indent=2, ensure_ascii=False), encoding="utf-8")
    print(caminho_json)

print("\nArquivos exportados:")
for caminho in ARQUIVOS_EXPORTADOS:
    print("-", caminho)


## 9. Critérios para decisão

A configuração só deve avançar para o loader de produção quando atender a estes critérios:

1. **Memory Pool:** `backend_real` deve corresponder ao backend solicitado em todas as repetições consideradas.
2. **Threads + Dictionary:** a combinação deve reduzir USS de forma repetível sem incompatibilidade de dados ou custo de tempo inaceitável.
3. **Parquets separados:** exigir equivalência de linhas, colunas e hashes em ESC e PROVA; verificar se a redução aparece nas duas referências.
4. **Downcasting:** avaliar a redução de `deep` e aceitar `float32` somente se o erro máximo ficar dentro da tolerância definida para o dashboard.
5. **Gunicorn:** observar RSS/USS da árvore de processos, não apenas do master; manter um worker enquanto o limite de memória for restrito.
6. **Repetibilidade:** se as faixas mínimo/máximo forem amplas ou houver inversões entre repetições, aumentar o número de execuções antes de decidir.

### Observação sobre o limite do Render

Não use somente USS para inferir se a aplicação caberá no limite de memória do serviço. Registre também RSS, número de processos e o comportamento durante callbacks. O consumo da aplicação completa pode incluir bibliotecas, servidor, caches e dados que não aparecem no worker isolado.

### Referências técnicas

- [PyArrow — `read_table`](https://arrow.apache.org/docs/python/generated/pyarrow.parquet.read_table.html): `use_threads` e `read_dictionary`.
- [Apache Arrow — Memory Management](https://arrow.apache.org/docs/python/memory.html): Memory Pool e backends.
- [Apache Arrow — Environment Variables](https://arrow.apache.org/docs/cpp/env_vars.html): `ARROW_DEFAULT_MEMORY_POOL`.
